# CBT Cognitive Distortion Classification with DeBERTa + HQDE\n\n**Optimized for Kaggle: 2x T4 GPUs, 4 vCPUs**\n**Using Real Conversational Dataset with Reasoning Chains**\n\nThis notebook trains an ensemble of DeBERTa models using the HQDE framework to classify cognitive distortions in therapy conversations.\n\n## 📁 Dataset Required\n**You must upload `cbt_therapy_conversations_full.json` to Kaggle:**\n1. Click **'Add Data'** (right sidebar)\n2. Click **'Upload'** → **'New Dataset'**\n3. Upload `cbt_therapy_conversations_full.json`\n4. Name it: **'cbt-therapy-conversations'**\n5. Add it to this notebook\n\n## Hardware Configuration\n- **GPUs**: 2x T4\n- **vCPUs**: 4\n- **RAM**: ~30GB\n\n## Dataset\n- **100 real therapy conversations** with reasoning chains\n- **10 cognitive distortion classes** (balanced)\n- **Full reasoning chains** (6 steps per conversation)\n\n## Expected Results\n- **Training Time**: ~30-45 minutes\n- **Expected Accuracy**: 85-92%

In [ ]:
# Install required packages\n!pip install -q transformers>=4.30.0 datasets>=2.14.0 accelerate>=0.20.0\n!pip install -q scikit-learn>=1.3.0 pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0

In [ ]:
import osimport jsonimport randomimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom tqdm.auto import tqdmimport warningswarnings.filterwarnings('ignore')import torchimport torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderfrom torch.cuda.amp import autocast, GradScalerfrom transformers import (    AutoTokenizer,    AutoModel,    AutoConfig,    get_cosine_schedule_with_warmup)from sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    accuracy_score,    classification_report,    confusion_matrix,    f1_score)print(f"PyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"CUDA version: {torch.version.cuda}")    print(f"Number of GPUs: {torch.cuda.device_count()}")    for i in range(torch.cuda.device_count()):        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Set random seedsdef set_seed(seed=42):    random.seed(seed)    np.random.seed(seed)    torch.manual_seed(seed)    torch.cuda.manual_seed_all(seed)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = Falseset_seed(42)# Configurationclass Config:    model_name = "microsoft/deberta-v3-base"    num_classes = 10    max_length = 256        num_workers = 4    batch_size = 8    num_epochs = 15    learning_rate = 2e-5    weight_decay = 0.01    warmup_ratio = 0.1    max_grad_norm = 1.0        dropout_rates = [0.1, 0.15, 0.2, 0.25]    learning_rates = [1.5e-5, 2e-5, 2.5e-5, 3e-5]        num_gpus = 2    device = "cuda" if torch.cuda.is_available() else "cpu"    use_amp = True        test_size = 0.2    val_size = 0.1        output_dir = "./cbt_output"        distortion_names = [        "All-or-Nothing Thinking",        "Overgeneralization",        "Mental Filter",        "Disqualifying the Positive",        "Jumping to Conclusions",        "Magnification/Catastrophizing",        "Emotional Reasoning",        "Should Statements",        "Labeling",        "Personalization"    ]config = Config()os.makedirs(config.output_dir, exist_ok=True)print(f"✓ Configuration set")print(f"  Model: {config.model_name}")print(f"  Ensemble workers: {config.num_workers}")print(f"  Effective batch size: {config.batch_size * config.num_gpus}")print(f"  Epochs: {config.num_epochs}")

In [ ]:
# Load real CBT therapy conversations datasetprint("Loading real CBT therapy conversations dataset...")# Find the uploaded dataset filepossible_paths = [    "/kaggle/input/cbt-therapy-conversations/cbt_therapy_conversations_full.json",    "/kaggle/input/cbt-therapy-dataset/cbt_therapy_conversations_full.json",    "/kaggle/input/cbt-dataset/cbt_therapy_conversations_full.json",    "./cbt_therapy_conversations_full.json",]dataset_path = Nonefor path in possible_paths:    if os.path.exists(path):        dataset_path = path        breakif dataset_path is None:    raise FileNotFoundError(        "Dataset file not found! Please upload 'cbt_therapy_conversations_full.json' to Kaggle.\n"        "Steps:\n"        "1. Click 'Add Data' (right sidebar)\n"        "2. Click 'Upload' → 'New Dataset'\n"        "3. Upload the file\n"        "4. Add it to this notebook"    )print(f"✓ Found dataset at: {dataset_path}")# Load the JSON filewith open(dataset_path, 'r', encoding='utf-8') as f:    data = json.load(f)print(f"✓ Dataset loaded")print(f"  Total conversations: {data['dataset_info']['total_conversations']}")print(f"  Classes: {data['dataset_info']['classes']}")# Process conversationsconversations = []for conv in data['conversations']:    # Combine patient statements with reasoning chain    # This gives the model the full context including reasoning    text = f"{conv['patient_statement']} {conv['patient_response']} Reasoning: {' '.join(conv['reasoning_chain'])}"        conversations.append({        "id": conv['id'],        "text": text,        "label": conv['cognitive_distortion'],        "emotion": conv['patient_emotion'],        "context": conv['context'],        "distortion_name": conv['distortion_name']    })df = pd.DataFrame(conversations)print(f"\n✓ Dataset processed: {len(df)} conversations")print(f"  Classes: {df['label'].nunique()}")print(f"\nClass Distribution:")for label in sorted(df['label'].unique()):    count = (df['label'] == label).sum()    name = config.distortion_names[label]    print(f"  {label}: {name:35s} - {count} samples")# Show exampleprint(f"\n📝 Example Conversation:")example = df.iloc[0]print(f"  Class: {example['distortion_name']}")print(f"  Emotion: {example['emotion']}")print(f"  Text (first 200 chars): {example['text'][:200]}...")

In [ ]:
# Split datatrain_df, temp_df = train_test_split(    df,     test_size=config.test_size + config.val_size,    random_state=42,    stratify=df['label'])val_df, test_df = train_test_split(    temp_df,    test_size=config.test_size / (config.test_size + config.val_size),    random_state=42,    stratify=temp_df['label'])print(f"✓ Data split:")print(f"  Training: {len(train_df)} samples")print(f"  Validation: {len(val_df)} samples")print(f"  Test: {len(test_df)} samples")# Load tokenizertokenizer = AutoTokenizer.from_pretrained(config.model_name)print(f"\n✓ Tokenizer loaded: {config.model_name}")# Dataset classclass CBTDataset(Dataset):    def __init__(self, texts, labels, tokenizer, max_length):        self.texts = texts        self.labels = labels        self.tokenizer = tokenizer        self.max_length = max_length        def __len__(self):        return len(self.texts)        def __getitem__(self, idx):        text = str(self.texts[idx])        label = self.labels[idx]                encoding = self.tokenizer(            text,            add_special_tokens=True,            max_length=self.max_length,            padding='max_length',            truncation=True,            return_attention_mask=True,            return_tensors='pt'        )                return {            'input_ids': encoding['input_ids'].flatten(),            'attention_mask': encoding['attention_mask'].flatten(),            'labels': torch.tensor(label, dtype=torch.long)        }# Create datasetstrain_dataset = CBTDataset(train_df['text'].values, train_df['label'].values, tokenizer, config.max_length)val_dataset = CBTDataset(val_df['text'].values, val_df['label'].values, tokenizer, config.max_length)test_dataset = CBTDataset(test_df['text'].values, test_df['label'].values, tokenizer, config.max_length)# Create data loaderstrain_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)val_loader = DataLoader(val_dataset, batch_size=config.batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)print(f"✓ Datasets and loaders created")

In [ ]:
class DeBERTaClassifier(nn.Module):    def __init__(self, model_name, num_classes, dropout_rate=0.1):        super(DeBERTaClassifier, self).__init__()                self.config = AutoConfig.from_pretrained(model_name)        self.config.hidden_dropout_prob = dropout_rate        self.config.attention_probs_dropout_prob = dropout_rate                self.deberta = AutoModel.from_pretrained(model_name, config=self.config)        self.dropout = nn.Dropout(dropout_rate)        self.classifier = nn.Linear(self.config.hidden_size, num_classes)                nn.init.xavier_uniform_(self.classifier.weight)        nn.init.zeros_(self.classifier.bias)        def forward(self, input_ids, attention_mask):        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)        pooled_output = outputs.last_hidden_state[:, 0, :]        pooled_output = self.dropout(pooled_output)        logits = self.classifier(pooled_output)        return logitsprint("✓ Model architecture defined")

In [ ]:
class EnsembleWorker:    def __init__(self, worker_id, config, device):        self.worker_id = worker_id        self.config = config        self.device = device                self.dropout_rate = config.dropout_rates[worker_id % len(config.dropout_rates)]        self.learning_rate = config.learning_rates[worker_id % len(config.learning_rates)]                print(f"Worker {worker_id}: dropout={self.dropout_rate}, lr={self.learning_rate}, device={device}")                self.model = DeBERTaClassifier(config.model_name, config.num_classes, self.dropout_rate).to(device)        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=config.weight_decay)        self.criterion = nn.CrossEntropyLoss()        self.scaler = GradScaler() if config.use_amp else None                self.train_losses = []        self.val_losses = []        self.val_accuracies = []        def train_epoch(self, train_loader, scheduler):        self.model.train()        total_loss = 0        correct = 0        total = 0                pbar = tqdm(train_loader, desc=f"Worker {self.worker_id} Training")        for batch in pbar:            input_ids = batch['input_ids'].to(self.device)            attention_mask = batch['attention_mask'].to(self.device)            labels = batch['labels'].to(self.device)                        self.optimizer.zero_grad()                        if self.config.use_amp:                with autocast():                    logits = self.model(input_ids, attention_mask)                    loss = self.criterion(logits, labels)                                self.scaler.scale(loss).backward()                self.scaler.unscale_(self.optimizer)                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)                self.scaler.step(self.optimizer)                self.scaler.update()            else:                logits = self.model(input_ids, attention_mask)                loss = self.criterion(logits, labels)                loss.backward()                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)                self.optimizer.step()                        scheduler.step()                        total_loss += loss.item()            _, predicted = torch.max(logits, 1)            total += labels.size(0)            correct += (predicted == labels).sum().item()                        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100 * correct / total:.2f}%'})                return total_loss / len(train_loader), 100 * correct / total        def evaluate(self, val_loader):        self.model.eval()        total_loss = 0        correct = 0        total = 0                with torch.no_grad():            for batch in tqdm(val_loader, desc=f"Worker {self.worker_id} Validation"):                input_ids = batch['input_ids'].to(self.device)                attention_mask = batch['attention_mask'].to(self.device)                labels = batch['labels'].to(self.device)                                logits = self.model(input_ids, attention_mask)                loss = self.criterion(logits, labels)                                total_loss += loss.item()                _, predicted = torch.max(logits, 1)                total += labels.size(0)                correct += (predicted == labels).sum().item()                return total_loss / len(val_loader), 100 * correct / total        def predict(self, data_loader):        self.model.eval()        all_logits = []                with torch.no_grad():            for batch in data_loader:                input_ids = batch['input_ids'].to(self.device)                attention_mask = batch['attention_mask'].to(self.device)                logits = self.model(input_ids, attention_mask)                all_logits.append(logits.cpu())                return torch.cat(all_logits, dim=0)print("✓ Ensemble worker class defined")

In [ ]:
workers = []devices = [f"cuda:{i % config.num_gpus}" for i in range(config.num_workers)]print(f"Creating {config.num_workers} ensemble workers:\n")for i in range(config.num_workers):    worker = EnsembleWorker(i, config, devices[i])    workers.append(worker)print(f"\n✓ {config.num_workers} workers created")

In [ ]:
print(f"🚀 Starting training for {config.num_epochs} epochs\n")best_ensemble_acc = 0for epoch in range(config.num_epochs):    print(f"\n{'='*80}")    print(f"Epoch {epoch + 1}/{config.num_epochs}")    print(f"{'='*80}")        for worker in workers:        num_training_steps = len(train_loader) * config.num_epochs        num_warmup_steps = int(num_training_steps * config.warmup_ratio)                scheduler = get_cosine_schedule_with_warmup(            worker.optimizer,            num_warmup_steps=num_warmup_steps,            num_training_steps=num_training_steps        )                train_loss, train_acc = worker.train_epoch(train_loader, scheduler)        val_loss, val_acc = worker.evaluate(val_loader)                worker.train_losses.append(train_loss)        worker.val_losses.append(val_loss)        worker.val_accuracies.append(val_acc)                print(f"\nWorker {worker.worker_id}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")        print(f"\nEnsemble Evaluation:")    all_worker_logits = []        for worker in workers:        logits = worker.predict(val_loader)        all_worker_logits.append(logits)        ensemble_logits = torch.stack(all_worker_logits).mean(dim=0)    ensemble_preds = torch.argmax(ensemble_logits, dim=1).numpy()    true_labels = val_df['label'].values        ensemble_acc = accuracy_score(true_labels, ensemble_preds) * 100    ensemble_f1 = f1_score(true_labels, ensemble_preds, average='weighted') * 100        print(f"  Ensemble Accuracy: {ensemble_acc:.2f}%")    print(f"  Ensemble F1-Score: {ensemble_f1:.2f}%")        if ensemble_acc > best_ensemble_acc:        best_ensemble_acc = ensemble_acc        print(f"  ✓ New best!")print(f"\n✓ Training completed! Best validation accuracy: {best_ensemble_acc:.2f}%")

In [ ]:
print("\nFinal Evaluation on Test Set\n")test_worker_logits = []for worker in workers:    logits = worker.predict(test_loader)    test_worker_logits.append(logits)ensemble_test_logits = torch.stack(test_worker_logits).mean(dim=0)ensemble_test_preds = torch.argmax(ensemble_test_logits, dim=1).numpy()test_true_labels = test_df['label'].valuestest_acc = accuracy_score(test_true_labels, ensemble_test_preds) * 100test_f1 = f1_score(test_true_labels, ensemble_test_preds, average='weighted') * 100print(f"{'='*80}")print(f"FINAL TEST RESULTS")print(f"{'='*80}")print(f"Ensemble Test Accuracy: {test_acc:.2f}%")print(f"Ensemble Test F1-Score: {test_f1:.2f}%")print(f"\nClassification Report:")print(classification_report(test_true_labels, ensemble_test_preds, target_names=config.distortion_names, digits=3))cm = confusion_matrix(test_true_labels, ensemble_test_preds)plt.figure(figsize=(12, 10))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',             xticklabels=[f"{i}" for i in range(10)],            yticklabels=[f"{i}" for i in range(10)])plt.title('Confusion Matrix - DeBERTa HQDE Ensemble (Real Dataset)')plt.ylabel('True Label')plt.xlabel('Predicted Label')plt.tight_layout()plt.savefig(f'{config.output_dir}/confusion_matrix.png', dpi=300, bbox_inches='tight')plt.show()print(f"\n✓ Confusion matrix saved")print(f"\nPer-Class Accuracy:")for i in range(config.num_classes):    class_mask = test_true_labels == i    if class_mask.sum() > 0:        class_acc = (ensemble_test_preds[class_mask] == test_true_labels[class_mask]).mean() * 100        print(f"  {i}: {config.distortion_names[i]:35s} - {class_acc:.2f}%")print(f"\nIndividual Worker Test Accuracy:")for i, worker in enumerate(workers):    worker_logits = test_worker_logits[i]    worker_preds = torch.argmax(worker_logits, dim=1).numpy()    worker_acc = accuracy_score(test_true_labels, worker_preds) * 100    print(f"  Worker {i}: {worker_acc:.2f}%")print(f"\n{'='*80}")print(f"✅ COMPLETE!")print(f"{'='*80}")print(f"Best Validation Accuracy: {best_ensemble_acc:.2f}%")print(f"Final Test Accuracy: {test_acc:.2f}%")print(f"Final Test F1-Score: {test_f1:.2f}%")print(f"\nDataset: Real CBT Therapy Conversations with Reasoning")